# Teste inicial da base de dados — NCBI Pathogen Detection

Objetivo desse notebook: **importar a base e conferir se está tudo funcionando.**

1. O arquivo abre certo?
2. As colunas que a gente precisa estão lá?
3. Tem dado suficiente para trabalhar depois?

---
## Passo 0 — Bibliotecas


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from google.colab import files
import warnings
from sklearn.model_selection import GroupShuffleSplit


---

# Como importar
Os dados vêm daqui: **https://www.ncbi.nlm.nih.gov/pathogens/isolates**

No site (leva uns 2 minutos):

1. Abre o link (é o **Isolates Browser** do Pathogen Detection).
2. Em **Filters**, escolhe um **Organism group**.
   Sugestão pro primeiro teste: **Salmonella enterica** — é o grupo com mais
   resultado de teste de laboratório disponível.
3. Em **Choose columns**, marca estas duas colunas:
   - **AMR genotypes** → os genes de resistência que aquela bactéria tem
   - **AST phenotypes** → o resultado real do teste de laboratório
4. Clica em **Download** e baixa como tabela (TSV ou CSV).


---
## Passo 2 — Importar a base

A célula abaixo abre uma janelinha para você escolher o arquivo que baixou.

In [ ]:
envbiar = files.upload()
nome_arquivo = list(envbiar.keys())[0]

# Descobre sozinho se o arquivo é separado por TAB (TSV) ou por vírgula (CSV)
with open(nome_arquivo, encoding="utf-8", errors="ignore") as f:
    primeira_linha = f.readline()
separador = "\t" if primeira_linha.count("\t") > primeira_linha.count(",") else ","

df = pd.read_csv(nome_arquivo, sep=separador, low_memory=False)

print("Arquivo:", nome_arquivo)
print("Separador:", "TAB (TSV)" if separador == "\t" else "vírgula (CSV)")
print()
print(f"Linhas (isolados): {len(df):,}")
print(f"Colunas: {len(df.columns)}")
print(f"Colunas: {len(df.columns)}")

Saving neisseria-gonorrheae.csv to neisseria-gonorrheae.csv
Arquivo: neisseria-gonorrheae.csv
Separador: vírgula (CSV)

Linhas (isolados): 72,503
Colunas: 18
Colunas: 18


### Visualizando tabela

In [ ]:
print(df.head())

print(df.columns.tolist())

         #Organism group     Strain                 Isolate identifiers  \
0  Neisseria gonorrhoeae    FA 1090  "AE004969","ERS28491451","FA 1090"   
1  Neisseria gonorrhoeae  NCCP11945              "CP001050","NCCP11945"   
2  Neisseria gonorrhoeae       1291                       "1291","ABZF"   
3  Neisseria gonorrhoeae      35/02                      "35/02","ABZG"   
4  Neisseria gonorrhoeae      DGI18                      "ABZH","DGI18"   

  Serovar         Isolate           Create date Location Isolation source  \
0     NaN  PDT000004444.2  2011-02-17T22:24:00Z      NaN              NaN   
1     NaN  PDT000005014.2  2010-05-12T22:44:00Z      NaN              NaN   
2     NaN  PDT000005594.2  2010-10-15T22:45:00Z      NaN              NaN   
3     NaN  PDT000005595.2  2010-10-15T22:45:00Z      NaN              NaN   
4     NaN  PDT000005596.2  2010-10-15T22:45:00Z      NaN              NaN   

  Isolation type  Food origin     SNP cluster  Min-same  Min-diff  \
0            NaN 

In [ ]:
def tabela(dados, linhas=5, max_texto=45):
    def limpar(v):
        if pd.isna(v):
            return "—"
        t = str(v)
        return t[:max_texto] + " …" if len(t) > max_texto else t

    return (dados.head(linhas).style
        .hide(axis="index")
        .format(limpar)
        .set_table_styles([
            {"selector": "th", "props": "background:#A8D5E8; color:#14384A; font-weight:600;"
                                        "text-align:left; padding:11px 14px;"},
            {"selector": "td", "props": "background:#FFFFFF; color:#22333B; padding:10px 14px;"
                                        "text-align:left; border-top:1px solid #DCEAF2;"
                                        "white-space:nowrap;"},
            {"selector": "tbody tr:nth-child(even) td", "props": "background:#F4FAFD;"},
        ]))

tabela(df)

#Organism group,Strain,Isolate identifiers,Serovar,Isolate,Create date,Location,Isolation source,Isolation type,Food origin,SNP cluster,Min-same,Min-diff,BioSample,Assembly,AST phenotypes,AMR genotypes,Computed types
Neisseria gonorrhoeae,FA 1090,"""AE004969"",""ERS28491451"",""FA 1090""",—,PDT000004444.2,2011-02-17T22:24:00Z,—,—,—,—,—,—,—,SAMN02604088,GCA_000006845.1,—,"farB=COMPLETE,mtrA=COMPLETE,mtrC=COMPLETE,mtr …",—
Neisseria gonorrhoeae,NCCP11945,"""CP001050"",""NCCP11945""",—,PDT000005014.2,2010-05-12T22:44:00Z,—,—,—,—,—,—,—,SAMN02603475,GCA_000020105.1,—,"farB=COMPLETE,folP_R228S=POINT,gyrA_D95G=POIN …",—
Neisseria gonorrhoeae,1291,"""1291"",""ABZF""",—,PDT000005594.2,2010-10-15T22:45:00Z,—,—,—,—,PDS000092963.1,—,—,SAMN02595238,GCA_000156755.1,—,"aph(3')-Ia=COMPLETE,farB=COMPLETE,mtrA=COMPLE …",—
Neisseria gonorrhoeae,35/02,"""35/02"",""ABZG""",—,PDT000005595.2,2010-10-15T22:45:00Z,—,—,—,—,PDS000010945.1,—,—,SAMN02595239,GCA_000156775.1,—,"farB=COMPLETE,folP_R228S=POINT,gyrA_S91F=POIN …",—
Neisseria gonorrhoeae,DGI18,"""ABZH"",""DGI18""",—,PDT000005596.2,2010-10-15T22:45:00Z,—,—,—,—,PDS000009044.4,—,—,SAMN02595240,GCA_000156795.1,—,"farB=COMPLETE,mtrA=COMPLETE,mtrC=COMPLETE,mtr …",—


# Bloco 1 — Configuração e verificação se tem AST ( SEM ELA NÃO DA)

In [ ]:
ANTIBIOTICO_ALVO = "ciprofloxacin"
MIN_POR_GENE = 3

def achar(*palavras):
    for c in df.columns:
        if all(p in c.lower() for p in palavras):
            return c
    return None

COL_GENES   = achar("amr", "genotype")
COL_AST     = achar("ast", "phenotype")
COL_CLUSTER = achar("snp", "cluster") or achar("pds")

print("Colunas do arquivo:")
for c in df.columns:
    print("  -", c)

print()
print("genes  :", COL_GENES   or ">>> NÃO ENCONTRADA")
print("ast    :", COL_AST     or ">>> NÃO ENCONTRADA")
print("cluster:", COL_CLUSTER or ">>> NÃO ENCONTRADA")

if COL_GENES is None or COL_AST is None:
    print("\n>>> Sem essas duas colunas não dá pra montar o alvo nem as features.")
print("Coluna do cluster:", COL_CLUSTER or "NÃO ENCONTRADA")

Colunas do arquivo:
  - #Organism group
  - Strain
  - Isolate identifiers
  - Serovar
  - Isolate
  - Create date
  - Location
  - Isolation source
  - Isolation type
  - Food origin
  - SNP cluster
  - Min-same
  - Min-diff
  - BioSample
  - Assembly
  - AST phenotypes
  - AMR genotypes
  - Computed types

genes  : AMR genotypes
ast    : AST phenotypes
cluster: SNP cluster
Coluna do cluster: SNP cluster


# Bloco 2 — Criar o alvo (resistente = 1, sensível = 0)

In [ ]:
if COL_AST is None:
    raise SystemExit("Sem a coluna AST phenotypes — baixe de novo marcando ela em Choose columns.")

def extrair_resistencia(texto, antibiotico):
    if pd.isna(texto): return np.nan
    for item in str(texto).split(","):
        if "=" not in item: continue
        ab, status = item.split("=", 1)
        if ab.strip().lower() == antibiotico.lower():
            s = status.strip().upper()
            if s.startswith("R"): return 1
            if s.startswith("S") and "SDD" not in s: return 0
    return np.nan

df["target"] = df[COL_AST].apply(lambda x: extrair_resistencia(x, ANTIBIOTICO_ALVO))
print(df["target"].value_counts(dropna=False))

target
NaN    71775
1.0      475
0.0      253
Name: count, dtype: int64


In [ ]:
from collections import Counter

print("Linhas no arquivo      :", f"{len(df):,}")
print("Com AMR genotypes      :", f"{df[COL_GENES].notna().sum():,}")
print("Com AST phenotypes     :", f"{df[COL_AST].notna().sum():,}")
print(f"Com R/S para {ANTIBIOTICO_ALVO}:", f"{df['target'].notna().sum():,}")
print()

contagem = Counter()
for t in df[COL_AST].dropna():
    for item in str(t).split(","):
        if "=" in item:
            contagem[item.split("=", 1)[0].strip().lower()] += 1

if not contagem:
    print(">>> A coluna AST veio, mas está 100% vazia: essa espécie não tem teste de laboratório.")
else:
    print("Antibióticos que existem no arquivo (nome exato -> nº de isolados):")
    for ab, n in contagem.most_common(25):
        print(f"   {ab:<35} {n:,}")

Linhas no arquivo      : 72,503
Com AMR genotypes      : 72,503
Com AST phenotypes     : 794
Com R/S para ciprofloxacin: 728

Antibióticos que existem no arquivo (nome exato -> nº de isolados):
   ceftriaxone                         794
   ciprofloxacin                       773
   cefixime                            769
   tetracycline                        766
   azithromycin                        716
   spectinomycin                       714
   penicillin                          513
   zoliflodacin                        448
   benzylpenicillin                    252
   cefoxitin                           62
   cefotaxime                          58
   cefepime                            57
   gentamicin                          6
   erythromycin                        2
   ertapenem                           1


# Bloco 3 — Filtrar e montar a lista de genes

In [ ]:
d = df.dropna(subset=["target", COL_GENES]).copy()
d["target"] = d["target"].astype(int)

d["genes_list"] = d[COL_GENES].apply(
    lambda t: [g.split("=")[0].strip() for g in str(t).split(",") if g.split("=")[0].strip()])

print(f"Isolados usáveis: {len(d)}  ({100*d['target'].mean():.1f}% resistente)")

Isolados usáveis: 728  (65.2% resistente)


# Bloco 4 — Separar treino e teste por cluster


In [ ]:
grupos = d[COL_CLUSTER] if COL_CLUSTER else pd.Series(d.index, index=d.index)

# isolado sem SNP cluster é singleton: vira um grupo só dele
sozinhos = pd.Series("sozinho_" + d.index.astype(str), index=d.index)
grupos = grupos.fillna(sozinhos)

itr, ite = next(GroupShuffleSplit(1, test_size=.2, random_state=42).split(d, d["target"], groups=grupos))
treino, teste = d.iloc[itr], d.iloc[ite]

print(f"Treino: {len(treino)} | Teste: {len(teste)}")
print(f"Isolados sem cluster: {d[COL_CLUSTER].isna().sum() if COL_CLUSTER else 0}")

Treino: 585 | Teste: 143
Isolados sem cluster: 164


# Bloco 5 — Montar a matriz de genes

In [ ]:
mlb = MultiLabelBinarizer().fit(treino["genes_list"])     # fit só no treino

def matriz(sub):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return pd.DataFrame(mlb.transform(sub["genes_list"]),
                            columns=[f"gene_{g}" for g in mlb.classes_], index=sub.index)

X_train, X_test = matriz(treino), matriz(teste)
manter = X_train.columns[X_train.sum() >= MIN_POR_GENE]
X_train, X_test = X_train[manter], X_test[manter]
y_train, y_test = treino["target"], teste["target"]
print(f"Features: {X_train.shape[1]}")

Features: 58


# Bloco 6 — Conferência

In [ ]:
vazou = len(set(grupos.iloc[itr]) & set(grupos.iloc[ite]))
print("Clusters em treino E teste:", vazou, "<- tem que ser 0")
print(f"Treino {X_train.shape} | Teste {X_test.shape}")
print(f"Resistentes -> treino: {y_train.sum()} | teste: {y_test.sum()}")

Clusters em treino E teste: 0 <- tem que ser 0
Treino (585, 58) | Teste (143, 58)
Resistentes -> treino: 387 | teste: 88
